# Lab 06 - Seam Carving (Redimensionamento com Reconhecimento de Conteudo)

Baseado em **Avidan & Shamir, *Seam Carving for Content-Aware Image Resizing* (SIGGRAPH 2007)**.

Biblioteca: https://pypi.org/project/seam-carving/

Funcionalidades implementadas:
1. **Reduzir** a imagem em x e y (remocao de seams).
2. **(a) Aumentar** a imagem em x e y (insercao de seams).
3. **(b) Remover ou preservar objetos** usando mascaras.

## 1. Instalacao e imports

In [ ]:
!pip install -q seam-carving

In [ ]:
import glob
import os
import shutil

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seam_carving

plt.rcParams['figure.figsize'] = (12, 6)

## 2. Carregamento das imagens

As imagens usadas sao as mesmas da pasta `imagens`: **beach**, **bergen**, **ibiza** e **puppy**.

O codigo abaixo procura as imagens na pasta `imagens/`. No Google Colab, se elas
ainda nao estiverem la, sera solicitado o upload (basta enviar os arquivos com esses nomes).

In [ ]:
PASTA_IMAGENS = 'imagens'
NOMES = ['beach', 'bergen', 'ibiza', 'puppy']

def localizar_imagem(pasta, nome):
    """Procura um arquivo 'nome.*' (qualquer extensao) dentro de 'pasta'."""
    padroes = glob.glob(os.path.join(pasta, nome + '.*'))
    if not padroes:
        raise FileNotFoundError(
            f"Nao encontrei a imagem '{nome}' em '{pasta}/'. "
            f"Arquivos esperados: {nome}.jpg / {nome}.jpeg / {nome}.png")
    return padroes[0]

def carregar_imagem(caminho):
    img = cv2.imread(caminho)
    if img is None:
        raise ValueError(f'Falha ao ler a imagem: {caminho}')
    return img

os.makedirs(PASTA_IMAGENS, exist_ok=True)

try:
    from google.colab import files
    # Move para a pasta arquivos que ja foram enviados para o diretorio raiz
    arquivos_root = glob.glob('*.jp*g') + glob.glob('*.png')
    for arq in arquivos_root:
        nome_lower = arq.lower()
        for nome in NOMES:
            if nome in nome_lower:
                destino = nome + os.path.splitext(arq)[1]
                shutil.move(arq, os.path.join(PASTA_IMAGENS, destino))
                print(f'Movido: {arq} -> {os.path.join(PASTA_IMAGENS, destino)}')
                break

    # Verifica se ainda faltam imagens; se sim, pede upload
    faltando = [n for n in NOMES
                if not glob.glob(os.path.join(PASTA_IMAGENS, n + '.*'))]
    if faltando:
        print(f'Envie as imagens: {faltando}')
        enviados = files.upload()
        for nome_arq in enviados.keys():
            nome_lower = nome_arq.lower()
            destino = nome_arq
            for nome in NOMES:
                if nome in nome_lower:
                    destino = nome + os.path.splitext(nome_arq)[1]
                    break
            shutil.move(nome_arq, os.path.join(PASTA_IMAGENS, destino))
except ImportError:
    # Fora do Colab: usa a pasta local 'imagens' diretamente
    pass

img_beach  = carregar_imagem(localizar_imagem(PASTA_IMAGENS, 'beach'))
img_bergen = carregar_imagem(localizar_imagem(PASTA_IMAGENS, 'bergen'))
img_ibiza  = carregar_imagem(localizar_imagem(PASTA_IMAGENS, 'ibiza'))
img_puppy  = carregar_imagem(localizar_imagem(PASTA_IMAGENS, 'puppy'))

print('Imagens carregadas com sucesso!')
for nome, im in [('beach', img_beach), ('bergen', img_bergen),
                 ('ibiza', img_ibiza), ('puppy', img_puppy)]:
    print(f'{nome:7s}: {im.shape[1]}x{im.shape[0]}')

## 3. Funcoes auxiliares de exibicao

O OpenCV carrega as imagens em **BGR**; o matplotlib espera **RGB**.

In [ ]:
def mostrar(img_bgr, titulo=''):
    plt.figure()
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(f'{titulo}  ({img_bgr.shape[1]}x{img_bgr.shape[0]})')
    plt.axis('off')
    plt.show()

def comparar(img_orig, img_res, t_orig='Original', t_res='Resultado'):
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    ax[0].imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
    ax[0].set_title(f'{t_orig}  ({img_orig.shape[1]}x{img_orig.shape[0]})')
    ax[0].axis('off')
    ax[1].imshow(cv2.cvtColor(img_res, cv2.COLOR_BGR2RGB))
    ax[1].set_title(f'{t_res}  ({img_res.shape[1]}x{img_res.shape[0]})')
    ax[1].axis('off')
    plt.show()

def sobrepor_mascara(img_bgr, mask, cor=(0, 0, 255), alpha=0.5):
    """Mostra a mascara (booleana) sobreposta a imagem, para conferencia."""
    overlay = img_bgr.copy()
    overlay[mask] = cor
    return cv2.addWeighted(overlay, alpha, img_bgr, 1 - alpha, 0)

## 4. Funcoes de Seam Carving

As mesmas funcoes estao no arquivo `seamcarving.py`. A funcao `seam_carving.resize`
cobre os dois casos de redimensionamento: tamanho alvo **menor** remove seams (reduz),
tamanho alvo **maior** insere seams (aumenta).

In [ ]:
def redimensionar(img, x_scale=1.0, y_scale=1.0, energy_mode='backward',
                  order='width-first', keep_mask=None, step_ratio=0.5):
    """Reduz (escala < 1) ou aumenta (escala > 1) a imagem em x e/ou y."""
    h, w = img.shape[:2]
    new_size = (int(round(w * x_scale)), int(round(h * y_scale)))  # (largura, altura)
    return seam_carving.resize(
        img, new_size,
        energy_mode=energy_mode,   # 'backward' ou 'forward'
        order=order,               # 'width-first' ou 'height-first'
        keep_mask=keep_mask,       # regiao a preservar
        step_ratio=step_ratio)     # aumentos grandes (>50%) feitos em etapas

def remover_objeto(img, drop_mask, keep_mask=None):
    """Remove o objeto coberto por drop_mask, preservando keep_mask (opcional)."""
    return seam_carving.remove_object(img, drop_mask=drop_mask, keep_mask=keep_mask)

## (a) Reduzir a imagem em x e y

Comparacao com o redimensionamento tradicional (`cv2.resize`), que distorce o conteudo.

In [ ]:
# Reduz a largura para 60% (remove seams verticais)
beach_reduz = redimensionar(img_beach, x_scale=0.6, y_scale=1.0)
comparar(img_beach, beach_reduz, 'Original', 'Seam carving - largura 60%')

# Compara com escala tradicional (distorce as pessoas / barcos)
h, w = img_beach.shape[:2]
beach_cv = cv2.resize(img_beach, (int(w*0.6), h))
comparar(beach_cv, beach_reduz, 'cv2.resize (distorce)', 'Seam carving (preserva)')

In [ ]:
# Reduz nas DUAS direcoes ao mesmo tempo: largura 70% e altura 80%
ibiza_xy = redimensionar(img_ibiza, x_scale=0.7, y_scale=0.8)
comparar(img_ibiza, ibiza_xy, 'Original', 'Reducao em x e y')

## (a) Aumentar a imagem em x e y

Para aumentar, basta passar um fator de escala **maior que 1**. A biblioteca insere
seams na ordem de remocao para distribuir os pixels artificiais (Secao 4.3 do artigo).
Aumentos grandes (>50%) sao feitos em etapas via `step_ratio`.

In [ ]:
# Aumenta a largura para 140% (insere seams verticais)
ibiza_mais_x = redimensionar(img_ibiza, x_scale=1.4, y_scale=1.0)
comparar(img_ibiza, ibiza_mais_x, 'Original', 'Aumento em x (140%)')

In [ ]:
# Aumenta a altura para 140% (insere seams horizontais)
bergen_mais_y = redimensionar(img_bergen, x_scale=1.0, y_scale=1.4)
comparar(img_bergen, bergen_mais_y, 'Original', 'Aumento em y (140%)')

In [ ]:
# Aumenta nas DUAS direcoes: 130% em x e 120% em y
puppy_maior = redimensionar(img_puppy, x_scale=1.3, y_scale=1.2)
comparar(img_puppy, puppy_maior, 'Original', 'Aumento em x e y')

## Criacao de mascaras

Uma mascara e' uma imagem booleana do mesmo tamanho da imagem onde `True` marca
a regiao de interesse (objeto a remover ou a preservar).

Abaixo ha duas formas de criar mascaras:
- **Por coordenadas** (retangulo ou poligono) - sempre funciona e e' reproduzivel;
- **Desenhando interativamente** no Colab (celula opcional mais adiante).

In [ ]:
def mascara_retangulo(shape, x1, y1, x2, y2):
    """Cria mascara booleana com um retangulo True. shape = (altura, largura, ...)."""
    mask = np.zeros(shape[:2], dtype=bool)
    mask[y1:y2, x1:x2] = True
    return mask

def mascara_poligono(shape, pontos):
    """Cria mascara booleana a partir de um poligono (lista de pontos [x, y])."""
    mask = np.zeros(shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [np.array(pontos, dtype=np.int32)], 255)
    return mask > 127

## (b) Remover objeto usando mascara

Marcamos com uma mascara o objeto a remover (`drop_mask`). Seams sao removidos ate'
que toda a regiao marcada desapareca.

In [ ]:
# Remove o veleiro/catamara central da imagem de Ibiza.
# A mascara cobre o barco no centro-inferior da imagem.
h, w = img_ibiza.shape[:2]
drop = mascara_retangulo(img_ibiza.shape,
                         x1=int(w*0.50), y1=int(h*0.62),
                         x2=int(w*0.62), y2=int(h*0.82))

# Confere a regiao marcada (em vermelho)
mostrar(sobrepor_mascara(img_ibiza, drop), 'Regiao a remover (drop_mask)')

ibiza_sem_barco = remover_objeto(img_ibiza, drop)
comparar(img_ibiza, ibiza_sem_barco, 'Original', 'Objeto removido')

## (b) Preservar objeto usando mascara (`keep_mask`)

Ao reduzir a imagem, marcamos com `keep_mask` a regiao importante para que os seams
**nao** passem por ela, protegendo o objeto da distorcao.

In [ ]:
# Reduz a largura do beach para 55% PRESERVANDO a pessoa no centro.
h, w = img_beach.shape[:2]
keep = mascara_retangulo(img_beach.shape,
                         x1=int(w*0.42), y1=int(h*0.25),
                         x2=int(w*0.58), y2=int(h*0.95))
mostrar(sobrepor_mascara(img_beach, keep, cor=(0, 255, 0)), 'Regiao a preservar (keep_mask)')

# Sem preservar (a pessoa fica distorcida)
beach_sem_keep = redimensionar(img_beach, x_scale=0.55)
# Preservando a pessoa
beach_com_keep = redimensionar(img_beach, x_scale=0.55, keep_mask=keep)
comparar(beach_sem_keep, beach_com_keep, 'Sem keep_mask', 'Com keep_mask (preserva a pessoa)')

### (b) Remover preservando outro objeto

E' possivel remover um objeto (`drop_mask`) e, ao mesmo tempo, proteger outro (`keep_mask`).

In [ ]:
# Remove um dos veleiros pequenos a direita, preservando o catamara central.
h, w = img_ibiza.shape[:2]
drop2 = mascara_retangulo(img_ibiza.shape,
                          x1=int(w*0.66), y1=int(h*0.48),
                          x2=int(w*0.72), y2=int(h*0.60))
keep2 = mascara_retangulo(img_ibiza.shape,
                          x1=int(w*0.50), y1=int(h*0.62),
                          x2=int(w*0.62), y2=int(h*0.82))
vis = sobrepor_mascara(img_ibiza, drop2, cor=(0, 0, 255))
vis = sobrepor_mascara(vis, keep2, cor=(0, 255, 0))
mostrar(vis, 'Vermelho = remover | Verde = preservar')

ibiza_dois = remover_objeto(img_ibiza, drop2, keep_mask=keep2)
comparar(img_ibiza, ibiza_dois, 'Original', 'Veleiro removido, catamara preservado')

## (Opcional) Desenhar a mascara interativamente no Colab

Execute a celula abaixo para abrir um quadro onde voce desenha (com o mouse) a regiao
da mascara sobre a imagem. Ao terminar, clique em **Salvar mascara**.
A mascara desenhada fica disponivel na variavel `mascara_desenhada`.

> Funciona apenas no Google Colab.

In [ ]:
from IPython.display import display, HTML
import base64

def desenhar_mascara(img_bgr, pincel=25):
    """Abre um canvas no Colab para desenhar a mascara. Retorna mascara booleana."""
    try:
        from google.colab.output import eval_js
    except ImportError:
        raise RuntimeError('Esta funcao so funciona no Google Colab.')
    h, w = img_bgr.shape[:2]
    _, buf = cv2.imencode('.png', img_bgr)
    b64 = base64.b64encode(buf).decode()
    html = '''
    <canvas id="cv" width="%d" height="%d" style="border:1px solid #888;cursor:crosshair"></canvas><br>
    <button id="salvar">Salvar mascara</button>
    <span id="st" style="margin-left:10px">Desenhe sobre o objeto...</span>
    <script>
      var cv = document.getElementById('cv'), ctx = cv.getContext('2d');
      var img = new Image();
      img.onload = function(){ ctx.drawImage(img,0,0); };
      img.src = 'data:image/png;base64,%s';
      var draw=false;
      ctx.strokeStyle='rgba(255,0,0,0.6)'; ctx.lineWidth=%d; ctx.lineCap='round';
      cv.onmousedown=function(e){draw=true; ctx.beginPath(); ctx.moveTo(e.offsetX,e.offsetY);};
      cv.onmousemove=function(e){ if(draw){ ctx.lineTo(e.offsetX,e.offsetY); ctx.stroke(); } };
      cv.onmouseup=function(){draw=false;}; cv.onmouseleave=function(){draw=false;};
      var saida=null;
      document.getElementById('salvar').onclick=function(){
        var m=document.createElement('canvas'); m.width=%d; m.height=%d;
        var mc=m.getContext('2d'); mc.drawImage(cv,0,0);
        var d=mc.getImageData(0,0,%d,%d).data;
        var out=document.createElement('canvas'); out.width=%d; out.height=%d;
        var oc=out.getContext('2d'); var od=oc.createImageData(%d,%d);
        // pixels desenhados (vermelho forte) viram brancos; resto preto
        for(var i=0;i<d.length;i+=4){
          var on=(d[i]>120 && d[i+1]<120 && d[i+2]<120)?255:0;
          od.data[i]=on; od.data[i+1]=on; od.data[i+2]=on; od.data[i+3]=255;
        }
        oc.putImageData(od,0,0); saida=out.toDataURL('image/png');
        document.getElementById('st').innerText='Mascara salva! Volte ao Python.';
      };
      window.getMaskData=function(){ return saida; };
    </script>
    ''' % (w, h, b64, pincel, w, h, w, h, w, h, w, h)
    display(HTML(html))
    input('Desenhe a mascara, clique em "Salvar mascara" e tecle ENTER aqui... ')
    data = eval_js('window.getMaskData()')
    if not data:
        raise RuntimeError('Nenhuma mascara foi salva. Clique em "Salvar mascara".')
    raw = base64.b64decode(data.split(',')[1])
    arr = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_GRAYSCALE)
    return arr > 127

# Exemplo de uso (descomente no Colab):
# mascara_desenhada = desenhar_mascara(img_ibiza)
# resultado = remover_objeto(img_ibiza, mascara_desenhada)
# comparar(img_ibiza, resultado, 'Original', 'Objeto removido (mascara desenhada)')

## Resumo

| Funcionalidade | Funcao | Parametro chave |
|---|---|---|
| Reduzir em x/y | `redimensionar` | escala < 1 |
| **(a)** Aumentar em x/y | `redimensionar` | escala > 1 (insere seams) |
| **(b)** Remover objeto | `remover_objeto` | `drop_mask` |
| **(b)** Preservar objeto | `redimensionar` / `remover_objeto` | `keep_mask` |